In [3]:
# llm_tool_retriever.py
"""
🤖 LLM Tool için Optimize Edilmiş Async RAPTOR Retrieve Fonksiyonu

Test sonuçlarına göre optimize edilmiş ayarlarla:
- balanced profile: 79.4 q/s peak performance
- 16 concurrent operations: optimal parallelization 
- Cache optimizations: %90+ hit rate sonrası
"""

import asyncio
import os
import logging
from typing import List, Dict, Any, Optional
import json
from nest_asyncio import apply
apply()  # Asyncio event loop için gerekli
# RAPTOR imports
from raptor import RetrievalAugmentation, RetrievalAugmentationConfig
from raptor.EmbeddingModels import CustomEmbeddingModel
from raptor.SummarizationModels import GPT41SummarizationModel

# Setup
os.environ["OPENAI_API_KEY"] = "None"
logging.basicConfig(level=logging.WARNING)  # Sadece hatalar için


class RaptorRetriever:
    """
    🚀 LLM Tool için optimize edilmiş RAPTOR retriever
    
    Test sonuçlarına göre ayarlanmış optimal configuration:
    - Performance profile: "balanced" (79.4 q/s peak)
    - Concurrency: 16 operations (optimal throughput)
    - Cache: 4 saat TTL, 0.8 similarity threshold
    - Top-k: 8 (balanced profile optimal)
    - Max tokens: 3500 (yeterli context)
    """
    
    def __init__(self, tree_path: str = "vectordb/raptor-production"):
        """
        Initialize RAPTOR with production-optimal settings
        
        Args:
            tree_path: RAPTOR tree dosya yolu
        """
        self.tree_path = tree_path
        self.ra = None
        self._initialize_raptor()
    
    def _initialize_raptor(self):
        """
        Test sonuçlarına göre optimize edilmiş RAPTOR configuration
        
        CONFIG AÇIKLAMALARI:
        - balanced profile: Speed vs Quality optimal balance (79.4 q/s)
        - max_concurrent_operations=16: Test'te optimal performans
        - cache_ttl=14400: 4 saat cache (production suitable)
        - tr_top_k=8: Balanced profile için optimal
        - tr_early_termination=True: Speed optimization
        - similarity_cache_threshold=0.8: Good cache hit rate
        """
        
        # Models
        embed_model = CustomEmbeddingModel()
        sum_model = GPT41SummarizationModel()
        
        # Test sonuçlarına göre OPTIMAL CONFIG
        config = RetrievalAugmentationConfig(
            # === MODELS ===
            embedding_model=embed_model,
            summarization_model=sum_model,
            tree_builder_type="cluster",
            
            # === PERFORMANCE OPTIMIZATION ===
            # Test'te 16 concurrent operations optimal throughput verdi
            enable_async=True,
            max_concurrent_operations=16,
            
            # === CACHE OPTIMIZATION ===
            # %90+ cache hit rate için optimize edilmiş
            enable_caching=True,
            cache_ttl=14400,  # 4 saat - production için ideal
            
            # === RETRIEVAL OPTIMIZATION ===
            tr_enable_caching=True,
            tr_adaptive_retrieval=True,
            tr_early_termination=True,    # Speed boost
            tr_top_k=8,                   # Balanced profile optimal
            tr_threshold=0.5,
            
            # === MONITORING ===
            enable_metrics=True,
            performance_monitoring=True,
            enable_progress_tracking=False,  # LLM tool için gereksiz noise
            
            # === TREE BUILDER CONFIG ===
            tb_batch_size=200,            # Büyük batch'ler efficient
            tb_build_mode="async",
            tb_enable_progress_tracking=False,
        )
        
        try:
            self.ra = RetrievalAugmentation(config=config, tree=self.tree_path)
        except Exception as e:
            raise RuntimeError(f"RAPTOR initialization failed: {e}")
    
    async def retrieve_batch_json(
        self, 
        queries: List[str],
        timeout_per_query: float = 30.0
    ) -> Dict[str, str]:
        """
        🎯 ANA LLM TOOL FONKSİYONU
        
        Query listesini alıp JSON formatında sonuç döner.
        Test sonuçlarına göre optimize edilmiş ayarlarla.
        
        Args:
            queries: Retrieve edilecek query'lerin listesi
            timeout_per_query: Her query için maksimum bekleme süresi
            
        Returns:
            Dict[str, str]: {"query": "retrieved_context", ...} formatında
            
        PERFORMANCE EXPECTATIONS:
        - Cold cache: ~18-36 queries/sec
        - Warm cache: ~60-80 queries/sec  
        - Cache hit rate: %70-90+
        - Context size: ~50-70KB per query
        """
        
        if not queries:
            return {}
        
        # Semaphore ile optimal concurrency control
        semaphore = asyncio.Semaphore(16)  # Test sonucuna göre optimal
        
        async def retrieve_single(query: str) -> tuple[str, str]:
            """Tek query'yi retrieve et"""
            async with semaphore:
                try:
                    # RAPTOR retrieve with optimal settings
                    context = await asyncio.wait_for(
                        asyncio.to_thread(
                            self.ra.retrieve,
                            query,
                            top_k=8,                    # Balanced profile optimal  
                            max_tokens=3500,            # Yeterli context
                            collapse_tree=False,        # Better quality
                            return_layer_information=False,  # LLM tool için gereksiz
                            use_async=True
                        ),
                        timeout=timeout_per_query
                    )
                    return query, context
                except Exception as e:
                    # Error handling - LLM tool için önemli
                    error_msg = f"Retrieve failed: {str(e)}"
                    return query, error_msg
        
        # Tüm query'leri paralel olarak işle
        tasks = [retrieve_single(query) for query in queries]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # JSON formatına çevir
        result_dict = {}
        for result in results:
            if isinstance(result, tuple):
                query, context = result
                result_dict[query] = context
            else:
                # Exception handling
                result_dict[f"error_{len(result_dict)}"] = f"Unexpected error: {result}"
        
        return result_dict


# === GLOBAL INSTANCE (LLM Tool için) ===
_retriever_instance: Optional[RaptorRetriever] = None

def get_retriever() -> RaptorRetriever:
    """
    Singleton pattern - LLM tool'da reuse için
    İlk çağrıda initialize, sonrakiler cache'den
    """
    global _retriever_instance
    if _retriever_instance is None:
        _retriever_instance = RaptorRetriever()
    return _retriever_instance


# === ANA LLM TOOL FONKSİYONU ===
async def retrieve_contexts(
    queries: List[str],
    tree_path: str = "vectordb/raptor-production",
    timeout_per_query: float = 30.0
) -> Dict[str, str]:
    """
    🤖 LLM TOOL İÇİN ANA FONKSİYON
    
    Test sonuçlarına göre optimize edilmiş RAPTOR retrieve fonksiyonu.
    
    Args:
        queries: Retrieve edilecek query'lerin listesi
        tree_path: RAPTOR tree dosya yolu
        timeout_per_query: Her query için timeout (saniye)
        
    Returns:
        Dict[str, str]: {
            "query1": "retrieved context for query1",
            "query2": "retrieved context for query2",
            ...
        }
        
    PERFORMANCE SPECS:
    - Concurrent Operations: 16 (test sonucuna göre optimal)
    - Profile: balanced (79.4 q/s peak performance)
    - Cache: 4 saat TTL, otomatik similarity matching
    - Expected Throughput: 18-80 q/s (cache durumuna göre)
    - Context Quality: ~50-70KB per query
    
    EXAMPLE:
        queries = [
            "What are the main findings?",
            "What methodology was used?",
            "What are the conclusions?"
        ]
        
        results = await retrieve_contexts(queries)
        # {
        #   "What are the main findings?": "The main findings include...",
        #   "What methodology was used?": "The methodology involved...",
        #   "What are the conclusions?": "The conclusions state..."
        # }
    """
    
    try:
        retriever = get_retriever()
        return await retriever.retrieve_batch_json(queries, timeout_per_query)
    except Exception as e:
        # LLM tool için error handling
        error_result = {}
        for query in queries:
            error_result[query] = f"Retrieval system error: {str(e)}"
        return error_result


# === CONVENIENCE WRAPPER (Sync version) ===
def retrieve_contexts_sync(
    queries: List[str],
    tree_path: str = "vectordb/raptor-production",
    timeout_per_query: float = 30.0
) -> Dict[str, str]:
    """
    Sync wrapper for LLM tools that don't support async
    
    Args:
        queries: Query listesi
        tree_path: RAPTOR tree path
        timeout_per_query: Timeout per query
        
    Returns:
        Dict[str, str]: Query-result mapping
    """
    return asyncio.run(retrieve_contexts(queries, tree_path, timeout_per_query))


# === EXAMPLE USAGE ===
async def main():
    """Örnek kullanım"""
    
    # Test query'leri
    test_queries = [
        "What is the main topic of this document?",
        "What are the key findings?", 
        "What methodology was used?",
        "What are the conclusions?",
        "What limitations are discussed?"
    ]
    
    print("🚀 Testing LLM Tool Retrieve Function...")
    print(f"Queries: {len(test_queries)}")
    
    # Ana fonksiyonu test et
    start_time = asyncio.get_event_loop().time()
    results = await retrieve_contexts(test_queries)
    end_time = asyncio.get_event_loop().time()
    
    # Sonuçları göster
    print(f"\n📊 RESULTS:")
    print(f"⏱️ Total time: {end_time - start_time:.2f}s")
    print(f"📝 Successful results: {len(results)}")
    print(f"🚀 Throughput: {len(test_queries) / (end_time - start_time):.1f} q/s")
    
    # JSON format örneği
    print(f"\n📋 JSON RESULT SAMPLE:")
    for i, (query, result) in enumerate(results.items()):
        if i < 2:  # İlk 2 sonucu göster
            print(f'"{query}": "{result[:100]}..."')
        else:
            break
    
    # JSON olarak kaydet (opsiyonel)
    with open("retrieve_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print("\n✅ Results saved to retrieve_results.json")
    return results


if __name__ == "__main__":
    # Test çalıştır
    result = asyncio.run(main())

2025-06-19 08:20:20,560 - Use pytorch device_name: cuda:0
2025-06-19 08:20:20,560 - Load pretrained SentenceTransformer: intfloat/multilingual-e5-large


🚀 Testing LLM Tool Retrieve Function...
Queries: 5


2025-06-19 08:20:30,758 - Successfully loaded tree from vectordb/raptor-production
2025-06-19 08:20:30,759 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'o200k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <raptor.SummarizationModels.GPT41SummarizationModel object at 0x731dd1609930>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.AsyncCustomEmbeddingModel object at 0x731dadfc8070>}
            Cluster Embedding Model: EMB
            Build Mode: async
            Batch Size: 200
            Max Concurrent Embeddings: 10
            Max Concurrent Summarizations: 5
            Progress Tracking: False
            Performance Monitoring: True
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Cluste


📊 RESULTS:
⏱️ Total time: 10.28s
📝 Successful results: 5
🚀 Throughput: 0.5 q/s

📋 JSON RESULT SAMPLE:
"What is the main topic of this document?": "Condensed synopsis of the material supplied   (ordered as in the original note, keeping most numeric..."
"What are the key findings?": "Summary of the analysis  1. Purpose      • Test how well different retrieval methods supply backgrou..."

✅ Results saved to retrieve_results.json


In [4]:
result

{'What is the main topic of this document?': 'Condensed synopsis of the material supplied   (ordered as in the original note, keeping most numerical and topical specifics).  1. ACL 2022 – Volume 1 (Long Papers)      • ≈ 300 long papers.      • Recurring themes:        – Scaling laws & very-large LMs (LoRA, prompt/instruction tuning).        – Efficient training / inference (quantisation, pruning, distillation).        – Long-document modelling (hierarchical attention, sparse transformers).        – Multilingual / low-resource NLP (massively-multilingual MT, zero-shot transfer).        – Societal issues (bias, fairness, green NLP, interpretability).        – Grounded generation (vision-language, code, retrieval-augmented dialogue).      • Representative numbers:        – Low-rank adaptation ≈ 95 % of full fine-tune quality with < 10 % trainable params.        – Sparse attention: 1.4–2 × speed-ups on 4 k-token inputs, negligible loss.        – Cross-lingual summarisation +3–4 ROUGE on XS